# ETL part

load EnterpriseRAG-Bench from Hugging Face

In [ ]:
# Step 1: Install required library
# "datasets" is the Hugging Face library used to download public datasets.
!pip install datasets -q

In [ ]:
# Step 2: Import required libraries

# load_dataset is used to load datasets from Hugging Face.
from datasets import load_dataset

# pandas is used to convert the dataset into table format.
import pandas as pd


# Step 3: Load EnterpriseRAG-Bench dataset from Hugging Face

# This dataset has two important parts:
# 1. documents  -> company-like documents used as RAG knowledge base
# 2. questions  -> test questions with expected answers and expected document IDs

documents_dataset = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "documents",
    split="test"
)

questions_dataset = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "questions",
    split="test"
)


# Step 4: Convert Hugging Face dataset into pandas DataFrame
# DataFrame means table format, like Excel/CSV.

documents_df = documents_dataset.to_pandas()
questions_df = questions_dataset.to_pandas()


# Step 5: Check how many rows and columns are present

print("Documents dataset shape:", documents_df.shape)
print("Questions dataset shape:", questions_df.shape)


# Step 6: Show column names
# This helps us understand what fields are available.

print("\nDocuments columns:")
print(documents_df.columns.tolist())

print("\nQuestions columns:")
print(questions_df.columns.tolist())


# Step 7: Preview first few rows

print("\nDocuments preview:")
display(documents_df.head())

print("\nQuestions preview:")
display(questions_df.head())

README.md:   0%|          | 0.00/6.93k [00:00<?, ?B/s]

data/documents/test.parquet:   0%|          | 0.00/1.41G [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

data/questions/test.parquet:   0%|          | 0.00/409k [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

Documents dataset shape: (511962, 4)
Questions dataset shape: (500, 7)

Documents columns:
['doc_id', 'source_type', 'title', 'content']

Questions columns:
['question_id', 'question_type', 'source_types', 'question', 'expected_doc_ids', 'gold_answer', 'answer_facts']

Documents preview:


,doc_id,source_type,title,content
0,dsid_e54ef48bae78474684a957cf613d47d5,confluence,Runbook: Deploy / Upgrade / Roll Back perf-can...,## Purpose\nThis runbook describes the operati...
1,dsid_229dd48e9b1d466a81ebaffe3ec84469,confluence,Cross-account GPU burst SLO contract and CI li...,Summary\n\nOverview:\nThis document defines th...
2,dsid_aeb0022d62bc43beb6549ba92e5655eb,confluence,Third-Party & Vendor Coordination Playbook for...,Overview\n\nPurpose:\nThis playbook documents ...
3,dsid_926174fc4900408c89c98abde46b7225,confluence,First Production Launch Checklist (Canonical T...,## Purpose\nThis page defines the **canonical ...
4,dsid_d511c6d8daf94f998ce6a3d97462af2d,confluence,Go-Live Runway and Week-3 Stability Playbook,Overview\n\nThis playbook defines the technica...



Questions preview:


,question_id,question_type,source_types,question,expected_doc_ids,gold_answer,answer_facts
0,qst_0001,basic,[github],What are the default size limits for file uplo...,[dsid_ae068ee4aa9640159427cd941bef0238],The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...
1,qst_0002,basic,[github],What is the name of the new metric added so SR...,[dsid_9550250a59e74f1bbd5612480b2e7100],The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...
2,qst_0003,basic,[linear],What are the acceptance criteria for the proje...,[dsid_3fd6af404fae48e6b8ea5a57875ef78f],The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...
3,qst_0004,basic,[fireflies],In the meeting about onboarding a SaaS product...,[dsid_6c4c1c875e704f09b4d791d64d7bc7e5],The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...
4,qst_0005,basic,[gmail],What failover sequence and recovery targets di...,[dsid_8e838ab6a98f4cbcb672d41f210ff89c],MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...


In [ ]:
# Step 8: Check one full document row
# This helps us understand what one knowledge-base document looks like.

print("One document example:")
display(documents_df.iloc[0])


# Step 9: Check one full question row
# This helps us understand what one test question looks like.

print("One question example:")
display(questions_df.iloc[0])

One document example:


,0
doc_id,dsid_e54ef48bae78474684a957cf613d47d5
source_type,confluence
title,Runbook: Deploy / Upgrade / Roll Back perf-can...
content,## Purpose\nThis runbook describes the operati...


One question example:


,0
question_id,qst_0001
question_type,basic
source_types,[github]
question,What are the default size limits for file uplo...
expected_doc_ids,[dsid_ae068ee4aa9640159427cd941bef0238]
gold_answer,The default limits are 10 MiB per file (max_fi...
answer_facts,[The default per file upload size limit (max_f...


In [ ]:
# Step 10: Select simple/basic questions for our first QA test set

# We are starting with "basic" questions because they are easier to verify.
# Later, we can test complex question types.

basic_questions_df = questions_df[questions_df["question_type"] == "basic"].copy()


# Step 11: Keep only questions that have exactly one expected document
# This makes the first version simple:
# one question -> one correct source document -> one expected answer.

basic_questions_df["expected_doc_count"] = basic_questions_df["expected_doc_ids"].apply(len)

single_doc_questions_df = basic_questions_df[
    basic_questions_df["expected_doc_count"] == 1
].copy()


# Step 12: Select first 10 questions
# We are choosing only 10 because this is the first QA prototype.
# After this works, we can increase the count.

selected_questions_df = single_doc_questions_df.head(10).copy()


# Step 13: Show selected questions

print("Selected questions count:", len(selected_questions_df))

display(
    selected_questions_df[
        [
            "question_id",
            "question_type",
            "question",
            "expected_doc_ids",
            "gold_answer",
            "answer_facts"
        ]
    ]
)

Selected questions count: 10


,question_id,question_type,question,expected_doc_ids,gold_answer,answer_facts
0,qst_0001,basic,What are the default size limits for file uplo...,[dsid_ae068ee4aa9640159427cd941bef0238],The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...
1,qst_0002,basic,What is the name of the new metric added so SR...,[dsid_9550250a59e74f1bbd5612480b2e7100],The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...
2,qst_0003,basic,What are the acceptance criteria for the proje...,[dsid_3fd6af404fae48e6b8ea5a57875ef78f],The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...
3,qst_0004,basic,In the meeting about onboarding a SaaS product...,[dsid_6c4c1c875e704f09b4d791d64d7bc7e5],The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...
4,qst_0005,basic,What failover sequence and recovery targets di...,[dsid_8e838ab6a98f4cbcb672d41f210ff89c],MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...
5,qst_0006,basic,In the draft spec about extending a routing po...,[dsid_184be937d34a412ab5e61366d54d8ed6],The draft proposes this canonical v1 signal pr...,[The draft spec proposes a canonical v1 priori...
6,qst_0007,basic,In a rolling investigation of a model regressi...,[dsid_72ec4a9962ba43e88acd61abbba1052d],"In the first comparison run, the baseline buil...","[In the first comparison run, the older baseli..."
7,qst_0008,basic,"In the internal shiproom runner notes, what is...",[dsid_5fc2dba9f6ac4af2b49b4f546a4298d0],The rollback plan is considered verified in st...,"[In staging, a verified rollback requires a ti..."
8,qst_0009,basic,"In the EdgePath evaluation email thread, what ...",[dsid_85deb10a652742baaf28af6149600001],Redwood said it wouldn't match the competitor'...,[Redwood did not match the competitors 50 perc...
9,qst_0010,basic,How does the new alerting approach group model...,[dsid_c1a6a71323c04c1ba5445aadea340362],It adds a lightweight token_stage_cohort servi...,[The approach adds a lightweight token_stage_c...


In [ ]:
# Step 14: Get the matching source document for each selected question

# We create a lookup table using doc_id.
# This helps us quickly find a document by its document ID.

documents_lookup = documents_df.set_index("doc_id")


# Step 15: Create rows for our lightweight QA dataset

lightweight_rows = []

for index, question_row in selected_questions_df.iterrows():

    # Each selected question has exactly one expected document ID.
    expected_doc_id = question_row["expected_doc_ids"][0]

    # Find the matching document from documents_df using that expected_doc_id.
    matching_document = documents_lookup.loc[expected_doc_id]

    # Create one QA test row.
    lightweight_rows.append({
        "test_id": f"RAG_TC_{len(lightweight_rows) + 1:03d}",

        # Unique code created by us for easy tracking.
        "input_code": f"EVL-RAG-{len(lightweight_rows) + 1:03d}",

        # Question details from questions dataset.
        "question_id": question_row["question_id"],
        "question_type": question_row["question_type"],
        "question": question_row["question"],

        # Expected source document details.
        "expected_doc_ids": expected_doc_id,
        "expected_doc_title": matching_document["title"],
        "expected_source_type": matching_document["source_type"],
        "expected_doc_content": matching_document["content"],

        # Expected answer details.
        "gold_answer": question_row["gold_answer"],
        "answer_facts": question_row["answer_facts"],

        # These fields will be filled after running the RAG chatbot/model.
        "actual_retrieved_doc_ids": "",
        "actual_output": "",

        # These fields will be filled by our embedding evaluator later.
        "expected_embedding_code": "",
        "actual_embedding_code": "",
        "similarity_score": "",

        # Final QA result fields.
        "faithfulness_check": "Not Checked",
        "hallucination_check": "Not Checked",
        "match_status": "Not Run",
        "remarks": ""
    })


# Step 16: Convert rows into DataFrame

lightweight_rag_df = pd.DataFrame(lightweight_rows)


# Step 17: Preview the lightweight QA dataset

print("Lightweight QA dataset shape:", lightweight_rag_df.shape)

display(lightweight_rag_df.head())

Lightweight QA dataset shape: (10, 20)


,test_id,input_code,question_id,question_type,question,expected_doc_ids,expected_doc_title,expected_source_type,expected_doc_content,gold_answer,answer_facts,actual_retrieved_doc_ids,actual_output,expected_embedding_code,actual_embedding_code,similarity_score,faithfulness_check,hallucination_check,match_status,remarks
0,RAG_TC_001,EVL-RAG-001,qst_0001,basic,What are the default size limits for file uplo...,dsid_ae068ee4aa9640159427cd941bef0238,"add multipart/form-data handling, strict conte...",github,description:\nMotivation: users integrating to...,The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...,,,,,,Not Checked,Not Checked,Not Run,
1,RAG_TC_002,EVL-RAG-002,qst_0002,basic,What is the name of the new metric added so SR...,dsid_9550250a59e74f1bbd5612480b2e7100,introduce-server-timebox-and-idempotent-cancel...,github,description:\nContext: production customers ob...,The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...,,,,,,Not Checked,Not Checked,Not Run,
2,RAG_TC_003,EVL-RAG-003,qst_0003,basic,What are the acceptance criteria for the proje...,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Develop interactive tone derivatives and Kappa...,linear,description:\nObjective: Create a deterministi...,The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...,,,,,,Not Checked,Not Checked,Not Run,
3,RAG_TC_004,EVL-RAG-004,qst_0004,basic,In the meeting about onboarding a SaaS product...,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,GCP Marketplace onboarding + billing review (R...,fireflies,summary:\nRedwood and the GCP Marketplace team...,The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...,,,,,,Not Checked,Not Checked,Not Run,
4,RAG_TC_005,EVL-RAG-005,qst_0005,basic,What failover sequence and recovery targets di...,dsid_8e838ab6a98f4cbcb672d41f210ff89c,Regional fallback priorities & logs — post-cal...,gmail,['From: Rafael Mendes <rafael.mendes@redwoodin...,MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...,,,,,,Not Checked,Not Checked,Not Run,


In [ ]:
# Step 18: Save the lightweight QA dataset as a CSV file

# This CSV is our final ETL output.
# We will use this file for QA testing and embedding comparison later.

lightweight_rag_df.to_csv("lightweight_rag_qa_dataset.csv", index=False)


# Step 19: Confirm the file is saved

print("CSV file created successfully: lightweight_rag_qa_dataset.csv")

CSV file created successfully: lightweight_rag_qa_dataset.csv


Embedding evaluator part

In [ ]:
# Step 20: Install sentence-transformers
# This library gives us an embedding model.
# Embedding means converting text into numeric meaning code/vector.

!pip install sentence-transformers -q

In [ ]:
# Step 21: Load embedding model

# SentenceTransformer is used to convert text into embedding vectors.
from sentence_transformers import SentenceTransformer

# This is a small and commonly used embedding model.
# It converts sentence meaning into a numeric vector.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [ ]:
# Step 22: Create expected embedding fingerprint code from gold_answer

# hashlib is used to create a small unique fingerprint from the full embedding.
# This makes the code short and readable, while still being based on the full embedding.
import hashlib
import numpy as np


def create_embedding_fingerprint(text, prefix="EXP"):
    # If text is empty or missing, return empty code.
    if pd.isna(text) or str(text).strip() == "":
        return ""

    # Convert the full text meaning into an embedding vector.
    # This vector represents the meaning of the expected answer.
    embedding_vector = embedding_model.encode(str(text))

    # Round the full embedding vector slightly before hashing.
    # This keeps the fingerprint stable and avoids tiny decimal noise.
    rounded_vector = np.round(embedding_vector, 6)

    # Convert the full rounded vector into bytes.
    # Hashing needs byte format, so we convert the numeric vector into bytes.
    vector_bytes = rounded_vector.tobytes()

    # Create a short hash/fingerprint from the full embedding vector.
    # We use only the first 8 characters to keep the code simple.
    fingerprint = hashlib.sha256(vector_bytes).hexdigest()[:8]

    # Return a small readable embedding code.
    # Example: EXP-a91f23c8
    return f"{prefix}-{fingerprint}"


# Apply the embedding fingerprint generator to gold_answer.
# This creates a short expected embedding code based on the full expected answer meaning.
lightweight_rag_df["expected_embedding_code"] = lightweight_rag_df["gold_answer"].apply(
    lambda text: create_embedding_fingerprint(text, prefix="EXP")
)


# Show the generated expected embedding codes.
display(
    lightweight_rag_df[
        ["test_id", "gold_answer", "expected_embedding_code"]
    ].head()
)
``

SyntaxError: invalid syntax (196246205.py, line 48)